# Week 2 Day 1 — Agent Foundations

**Reasoning loops, tool calling & raw Python agents** using the **Groq** API. No LangChain / LangGraph / CrewAI.

The runnable agent is also in `agent.py`. API key goes in `.env`.

## Setup

```powershell
pip install -r requirements.txt
```

Requires `GROQ_API_KEY` in `.env`.


In [1]:
import os
import json
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path(".env"))
except ImportError:
    pass

from groq import Groq
from agent import (
    TOOLS,
    TOOL_SCHEMAS,
    MODEL,
    MAX_ITERATIONS,
    execute_tool,
    run_agent,
    calculator,
    get_weather,
    read_file,
    get_client,
)

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env before running API cells"
client = get_client()
print("Model:", MODEL)
print("Tools:", [t["function"]["name"] for t in TOOLS])

Model: openai/gpt-oss-20b
Tools: ['calculator', 'get_weather', 'read_file']


## Task 1 — Agent concepts & mental model

### Agent vs chatbot vs workflow

| | Chatbot | Workflow | Agent |
|--|---------|----------|-------|
| Control | User asks → model replies | Fixed steps A→B→C | Model chooses next action |
| Tools | Usually none | Pre-wired calls | Dynamic tool use |
| Planning | Single turn (mostly) | Author planned it | Multi-step, adapts to results |

**Agentic** behavior means some mix of: **autonomy**, **tool use**, **multi-step planning**, and **self-correction**.

### ReAct pattern

```
┌──────────┐     ┌──────────┐     ┌────────────┐
│  Reason  │ ──▶ │   Act    │ ──▶ │  Observe   │──┐
│  (think) │     │ (tool)   │     │ (result)   │  │
└──────────┘     └──────────┘     └────────────┘  │
      ▲                                           │
      └───────────────────────────────────────────┘
```

### When an agent is overkill

If the path is known — one lookup, one formula, or fixed ETL — use a script or single prompt. Agents add latency, cost, and failure modes.

## Task 2 — Tool calling fundamentals

Tools need **name**, **description**, and **parameters** (JSON schema). Descriptions matter because the model selects tools and fills arguments from that text.

Below: inspect schemas, smoke-test local tools, then a **manual** round-trip (`tool_calls` → execute → `role=tool`).

In [2]:
print(json.dumps(TOOL_SCHEMAS, indent=2))

print("\nLocal smoke tests:")
print(calculator("17 * 23"))
print(get_weather("Karachi"))
print(read_file("sample_notes.txt")[:200], "...")

[
  {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression. Use for any math (sums, differences, averages). Input is a single expression string such as '32 - 28' or '(14 + 22) / 2'.",
    "input_schema": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "Arithmetic expression using + - * / ** // % and parentheses"
        }
      },
      "required": [
        "expression"
      ]
    }
  },
  {
    "name": "get_weather",
    "description": "Look up current weather for one city. Call once per city. Known cities: Karachi, Lahore, Islamabad, London, Tokyo. Returns temperature in Celsius and a short condition.",
    "input_schema": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name, e.g. Karachi or Lahore"
        }
      },
      "required": [
        "city"
      ]
    }
  },
  {
    "name": "read_file",

In [3]:
messages = [
    {"role": "system", "content": "Use tools when helpful."},
    {"role": "user", "content": "What is 17 * 23? Use the calculator tool."},
]

response = client.chat.completions.create(
    model=MODEL,
    max_tokens=1024,
    tools=TOOLS,
    tool_choice="auto",
    messages=messages,
)
message = response.choices[0].message
print("content:", message.content)
print("tool_calls:", message.tool_calls)

assert message.tool_calls, "Expected the model to call calculator"

messages.append({
    "role": "assistant",
    "content": message.content,
    "tool_calls": [
        {
            "id": tc.id,
            "type": "function",
            "function": {"name": tc.function.name, "arguments": tc.function.arguments},
        }
        for tc in message.tool_calls
    ],
})

for tc in message.tool_calls:
    args = json.loads(tc.function.arguments or "{}")
    observation = execute_tool(tc.function.name, args)
    print("observation:", observation)
    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "name": tc.function.name,
        "content": observation,
    })

final = client.chat.completions.create(
    model=MODEL,
    max_tokens=1024,
    tools=TOOLS,
    messages=messages,
)
print("\nFinal:", final.choices[0].message.content)

content: None
tool_calls: [ChatCompletionMessageToolCall(id='fc_74924e46-8e6a-4934-afa2-95222f355ca4', function=Function(arguments='{"expression":"17 * 23"}', name='calculator'), type='function')]
observation: {"expression": "17 * 23", "result": 391.0}



Final: 17 × 23 = **391**


## Task 3 — Minimal agent loop

`run_agent` implements: send → if `tool_calls` execute → append tool results → repeat until final text, with **`max_iterations`**.

Multi-step test: weather in **two** cities + which is warmer (**2+** tool calls).

In [4]:
scratch = {}
prompt = (
    "Look up the weather in Karachi and Lahore, then use the calculator "
    "to compute the temperature difference. Which city is warmer?"
)
answer = run_agent(prompt, max_iterations=MAX_ITERATIONS, working_memory=scratch)
print("\n---")
print("iterations_used:", scratch.get("iterations_used"))
print("num_tool_calls:", len(scratch.get("tool_calls", [])))
print("tool_calls:", scratch.get("tool_calls"))
print("answer:", answer)
assert len(scratch.get("tool_calls", [])) >= 2, "Expected at least 2 tool calls"


=== USER ===
Look up the weather in Karachi and Lahore, then use the calculator to compute the temperature difference. Which city is warmer?

=== ITERATION ===
1/8



=== ACT (tool_call) ===
name=get_weather
id=fc_a661e8be-3ba1-4558-a6c0-380debed3407
input={"city": "Karachi"}

=== OBSERVE (tool_result) ===
{"city": "Karachi", "temp_c": 32, "condition": "humid"}

=== ITERATION ===
2/8



=== ACT (tool_call) ===
name=get_weather
id=fc_6a360311-9192-4520-a844-c82caa44cb4d
input={"city": "Lahore"}

=== OBSERVE (tool_result) ===
{"city": "Lahore", "temp_c": 28, "condition": "clear"}

=== ITERATION ===
3/8



=== ACT (tool_call) ===
name=calculator
id=fc_31c0ffc1-c9c2-4a48-908d-5c4e8903d626
input={"expression": "32 - 28"}

=== OBSERVE (tool_result) ===
{"expression": "32 - 28", "result": 4.0}

=== ITERATION ===
4/8

=== REASONING / TEXT ===
Karachi’s temperature is 32 °C, while Lahore’s is 28 °C.  
The difference is \(32 - 28 = 4\) °C, and **Karachi is the warmer city.**

=== FINAL ANSWER ===
Karachi’s temperature is 32 °C, while Lahore’s is 28 °C.  
The difference is \(32 - 28 = 4\) °C, and **Karachi is the warmer city.**

---
iterations_used: 4
num_tool_calls: 3
tool_calls: [{'name': 'get_weather', 'input': {'city': 'Karachi'}}, {'name': 'get_weather', 'input': {'city': 'Lahore'}}, {'name': 'calculator', 'input': {'expression': '32 - 28'}}]
answer: Karachi’s temperature is 32 °C, while Lahore’s is 28 °C.  
The difference is \(32 - 28 = 4\) °C, and **Karachi is the warmer city.**


## Task 4 — Memory & state + logging

- **Conversation memory**: the `messages` list (system / user / assistant / tool).
- **Working memory**: `scratch` tracks tool calls and observations for debugging.

The agent prints **REASONING / TEXT**, **ACT (tool_call)**, and **OBSERVE (tool_result)** each step.

In [5]:
print(json.dumps({
    "iterations_used": scratch.get("iterations_used"),
    "tool_calls": scratch.get("tool_calls"),
    "observations": scratch.get("observations"),
    "final": scratch.get("final"),
}, indent=2))

{
  "iterations_used": 4,
  "tool_calls": [
    {
      "name": "get_weather",
      "input": {
        "city": "Karachi"
      }
    },
    {
      "name": "get_weather",
      "input": {
        "city": "Lahore"
      }
    },
    {
      "name": "calculator",
      "input": {
        "expression": "32 - 28"
      }
    }
  ],
  "observations": [
    "{\"city\": \"Karachi\", \"temp_c\": 32, \"condition\": \"humid\"}",
    "{\"city\": \"Lahore\", \"temp_c\": 28, \"condition\": \"clear\"}",
    "{\"expression\": \"32 - 28\", \"result\": 4.0}"
  ],
  "final": "Karachi\u2019s temperature is 32\u202f\u00b0C, while Lahore\u2019s is 28\u202f\u00b0C.  \nThe difference is \\(32 - 28 = 4\\)\u202f\u00b0C, and **Karachi is the warmer city.**"
}


## Task 5 — Failure modes & guardrails

| Failure mode | Mitigation |
|--------------|------------|
| Infinite loops | `max_iterations` |
| Unknown tool | Only execute registered tools; return error JSON |
| Wrong arguments | JSON schema + catch missing keys |
| Tool runtime errors | Return error string as tool content; log every step |
| Ambiguous request | System prompt: admit limits |
| Missing capability | Honest refusal |

**Why frameworks?** Hand-built loops teach the model. LangChain / LangGraph / CrewAI package retries, memory, tracing, and multi-agent wiring.

In [6]:
failure_cases = [
    ("ambiguous", "Do the thing with the numbers for that city."),
    ("tool_error", "Get the weather in Atlantis."),
    ("missing_tool", "Send an email to alice@example.com saying hello."),
]

for name, prompt in failure_cases:
    print("\n" + "#" * 60)
    print("CASE:", name)
    mem = {}
    out = run_agent(prompt, max_iterations=4, working_memory=mem)
    print("RESULT:", out)
    print("tools_used:", mem.get("tool_calls"))


############################################################
CASE: ambiguous



=== USER ===
Do the thing with the numbers for that city.

=== ITERATION ===
1/4



=== REASONING / TEXT ===
Sure! Could you let me know which city you’re referring to and what calculation you’d like me to perform with the numbers? That way I can give you the exact result you need.

=== FINAL ANSWER ===
Sure! Could you let me know which city you’re referring to and what calculation you’d like me to perform with the numbers? That way I can give you the exact result you need.
RESULT: Sure! Could you let me know which city you’re referring to and what calculation you’d like me to perform with the numbers? That way I can give you the exact result you need.
tools_used: []

############################################################
CASE: tool_error



=== USER ===
Get the weather in Atlantis.

=== ITERATION ===
1/4



=== REASONING / TEXT ===
I’m sorry, but I can’t find weather information for Atlantis. The weather lookup tool only supports a limited set of known cities (Karachi, Lahore, Islamabad, London, Tokyo). If you need the weather for a different real location, just let me know the city name!

=== FINAL ANSWER ===
I’m sorry, but I can’t find weather information for Atlantis. The weather lookup tool only supports a limited set of known cities (Karachi, Lahore, Islamabad, London, Tokyo). If you need the weather for a different real location, just let me know the city name!
RESULT: I’m sorry, but I can’t find weather information for Atlantis. The weather lookup tool only supports a limited set of known cities (Karachi, Lahore, Islamabad, London, Tokyo). If you need the weather for a different real location, just let me know the city name!
tools_used: []

############################################################
CASE: missing_tool



=== USER ===
Send an email to alice@example.com saying hello.

=== ITERATION ===
1/4



=== REASONING / TEXT ===
I’m sorry, but I don’t have the ability to send emails.

=== FINAL ANSWER ===
I’m sorry, but I don’t have the ability to send emails.
RESULT: I’m sorry, but I don’t have the ability to send emails.
tools_used: []


## Summary

This notebook covers Tasks 1-5: concepts, tool schemas, the agent loop, memory/logging, and failure cases. Companion files: `agent.py` and `writeup.md`.
